# 第一章：练习答案

建议先做完[主教程](ch01.ipynb)再看。这份 Notebook 可独立运行。
前两题只用 Python；第三题会依次加载小模型两次，请先关闭主教程 kernel 或运行它末尾的释放格。

## 1. argmax 返回什么？

最大分数是 4.0，位于下标 0，所以选“甲”。下面用普通 Python 验证同一个选择规则。

In [1]:
tokens = ['甲', '乙', '丙']
scores = [4.0, 3.5, 0.8]
chosen_id = max(range(len(scores)), key=lambda i: scores[i])
assert chosen_id == 0
print('ID:', chosen_id, 'token:', tokens[chosen_id])

ID: 0 token: 甲


## 2. 第八轮输入长度

输入里包含 19 个原始 token 和之前生成的 7 个，共 26 个。第八个新 ID 是这一轮的输出。

In [2]:
lengths = [19 + i for i in range(8)]
assert lengths[-1] == 26
print('各轮输入长度:', lengths)

各轮输入长度: [19, 20, 21, 22, 23, 24, 25, 26]


## 3. 输出上限从 8 改为 12

权重、输入和计算方式不变，greedy 的每一步不依赖“将来还要生成几个”，所以共同前缀应相同。
若遇到 EOS，两次都可能提前结束，12 是上限。

这里运行从 `tutorial-01` 冻结的[原脚本](reference/generate.py)，分别保存 trace。
它和主教程显式写出的循环是同一路径。两次子进程按顺序执行，退出后释放模型内存。

In [3]:
import json
from pathlib import Path
import subprocess
import sys
import tempfile

import torch

here = Path.cwd().resolve()
chapter_dir = next(p for p in [here, here / 'ch01', here / 'notebooks/ch01']
                   if (p / 'reference/generate.py').is_file())
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('设备:', device)

设备: mps


In [4]:
traces = {}
with tempfile.TemporaryDirectory() as work_dir:
    for count in [8, 12]:
        path = Path(work_dir) / f'trace-{count}.json'
        result = subprocess.run([
            sys.executable, str(chapter_dir / 'reference/generate.py'),
            '--max-new-tokens', str(count), '--device', device,
            '--trace-json', str(path),
        ], capture_output=True, text=True, timeout=240)
        if result.returncode:
            raise RuntimeError(result.stderr[-4000:])
        traces[count] = json.loads(path.read_text())
        print(f'上限 {count}:', traces[count]['generated_ids'])

上限 8: [79852, 18887, 220, 98435, 98618, 103806, 95779, 103725]


上限 12: [79852, 18887, 220, 98435, 98618, 103806, 95779, 103725, 111892, 95726, 108777, 111734]


In [5]:
short = traces[8]['generated_ids']
long = traces[12]['generated_ids']
assert long[:len(short)] == short
historical = json.loads((chapter_dir / 'reference/chapter01-trace.json').read_text())
print('8 / 12 的共同前缀一致:', True)
print('与历史 8-token 记录一致:', short == historical['generated_ids'])

8 / 12 的共同前缀一致: True
与历史 8-token 记录一致: True


下一步回到主教程，确认你能在循环中指出三行关键代码：计算 logits、argmax、接回输入。
这里没有执行吞吐测试，也没有比较优化引擎的速度。